# 调试与兼容处理

学习目标：能区分样式失效原因，使用布局调试工具，并按目标浏览器建立可用的渐进增强与回退。

前置知识：层叠与继承、盒模型、Flexbox、Grid、媒体查询和键盘焦点。

适用范围：现代浏览器；开发者工具步骤以 Chrome 为例，面板名称和提示可能随版本调整。故意的无效值已在源码注释标明。

环境准备：[环境配置与运行](README.md)。

配套脚本：位于 scripts/20-debugging-and-compatibility/。

1. [index.html](scripts/20-debugging-and-compatibility/index.html)、[debug.css](scripts/20-debugging-and-compatibility/debug.css)：匹配、覆盖、无效声明和不适用声明。
2. [layout.html](scripts/20-debugging-and-compatibility/layout.html)、[layout.css](scripts/20-debugging-and-compatibility/layout.css)：盒模型、Flexbox 与 Grid 工具。
3. [enhancement.html](scripts/20-debugging-and-compatibility/enhancement.html)、[enhancement.css](scripts/20-debugging-and-compatibility/enhancement.css)：特性查询与可用回退。
4. [controls.html](scripts/20-debugging-and-compatibility/controls.html)、[controls.css](scripts/20-debugging-and-compatibility/controls.css)：前缀、控件及视口与键盘检查。

## 打开配套页面

Step 1：在已激活 Python 环境的终端中，从项目根目录进入本技术目录。

```bash
cd content/Web与应用开发/css
```

Step 2：启动本章预览服务。

```bash
python -m http.server 8101 --bind 127.0.0.1
```

Step 3：打开[本章示例首页](http://127.0.0.1:8101/scripts/20-debugging-and-compatibility/index.html)。

服务根目录是 content/Web与应用开发/css；修改文件后保存并刷新

Step 4：在服务终端按 Ctrl+C 停止服务。

## 1 区分未加载、未匹配、无效与被覆盖

先确认打开的是正确页面和最新文件。在 Network 中刷新，筛选 CSS，检查请求地址、状态、响应内容以及 Content-Type；状态成功却返回 HTML 错误页，也不能当作样式表正确加载。

在 Elements 选中目标元素，按以下顺序缩小范围：

（1）Styles 是否出现预期规则？没有时核对类名、祖先关系、媒体与特性条件；.missing 在本页没有匹配元素。

（2）声明是否有效？color: 20px 是故意的无效值，浏览器不会把长度当作颜色。

（3）是否被更强声明覆盖？.notice 的颜色声明语法正确，但 #notice 优先级更高。

（4）属性是否适用于当前盒？非替换普通行内盒的 width 不控制其布局宽度，属于声明有效却不产生该尺寸效果。

划线或警告标记可能表示不同原因；悬停查看工具提示，不要只看颜色猜测。括号、分号、选择器结构错误可能影响多条声明或整条规则，不能一概认为只丢一行。

```html
<p id="notice" class="notice">定位实际生效的颜色。</p>
<p class="invalid">故意保留一个无效值。</p>
<p><span class="inline-box">行内文字</span></p>
```

```css
.notice { color: teal; }
#notice { color: navy; }
.missing { color: maroon; }
.invalid { color: teal; color: 20px; }
.inline-box { display: inline; width: 160px; border: 1px solid; }
/* 四种情况：notice被覆盖、missing未匹配、20px无效、行内盒width有效但不适用。 */
```

配套文件：[index.html](scripts/20-debugging-and-compatibility/index.html)、[debug.css](scripts/20-debugging-and-compatibility/debug.css) · [浏览器预览](http://127.0.0.1:8101/scripts/20-debugging-and-compatibility/index.html)

## 2 本章使用的属性

| 完整属性名 | 中文名称／含义 | 用途或对象 |
| --- | --- | --- |
| color | 前景颜色 | 检查匹配与覆盖 |
| display | 盒生成和布局方式 | 核对当前布局上下文 |
| width | 宽度 | 比较声明与实际盒尺寸 |
| box-sizing | 尺寸计算方式 | 解释内容盒与边框盒 |
| gap | 间距 | 观察布局间隙 |
| grid-template-columns | 列轨道定义 | 检查Grid轨道 |
| appearance | 原生外观 | 控件样式与历史前缀示例 |
| outline | 轮廓 | 观察焦点 |
| overflow-wrap | 长文本换行策略 | 检查窄屏溢出 |

## 3 计算样式、盒模型和布局工具

Computed 汇总解析后的属性值，可展开属性追溯声明来源；Styles 更适合查看匹配规则、继承和覆盖链。工具面板可能展示解析值或最终布局相关值，不能把所有显示值都当作同一个规范阶段，更不能把 width 的数字直接当作元素外宽。

在盒模型图中分别检查内容、内边距、边框和外边距；本例显式使用 content-box，所以外宽要加两侧内边距与边框。

选中 .flex-demo，点击 Elements 中的 flex 徽章打开覆盖层；在 Styles 的 Flexbox 编辑器试调对齐，同时检查容器方向、换行和项目伸缩。仅在项目上改 justify-content 不会控制父容器的排列。

选中 .grid-demo，点击 grid 徽章，在 Layout 中打开线号和轨道尺寸等显示。核对实际轨道、gap 与可用空间，再判断溢出是否来自项目最小尺寸或长内容。覆盖层是调试辅助，不会修改源样式。

临时关闭一条声明验证原因，一次只改一个条件。确认有效后在本地 CSS 保存，再刷新复查；普通 DevTools 临时编辑不会自动写入项目。

```html
<div class="measure">内容宽度 180px 的盒子</div>
<div class="flex-demo"><span>第一项</span><span>第二项</span><span>第三项</span></div>
<div class="grid-demo"><span>第一格</span><span>第二格</span><span>第三格</span></div>
```

```css
.measure { box-sizing: content-box; width: 180px; padding: 12px; border: 3px solid; }
.flex-demo { display: flex; flex-wrap: wrap; gap: 12px; max-width: 340px; margin-block: 24px; }
.flex-demo > span { flex: 1 1 100px; background: #cce8ed; }
.grid-demo { display: grid; grid-template-columns: 1fr 2fr; gap: 12px; max-width: 340px; }
.grid-demo > span { background: #f6d69b; }
/* 盒子外宽=180+24+6=210px；Grid的1:2分配作用于扣除gap后的轨道空间。 */
```

配套文件：[layout.html](scripts/20-debugging-and-compatibility/layout.html)、[layout.css](scripts/20-debugging-and-compatibility/layout.css) · [浏览器预览](http://127.0.0.1:8101/scripts/20-debugging-and-compatibility/layout.html)

## 4 用 @supports 做渐进增强

@supports 是条件 @ 规则，(display: grid) 查询浏览器是否支持这条声明；and 要求两个条件成立，or 表示任一成立，not 否定条件。混合逻辑时按语法加括号分组，不写成随意串联的自然语言。

先写单列基础内容，再在支持条件中设置 Grid。只支持基础规则时，内容和阅读顺序仍完整；支持增强时获得列与间距。增强块中的窄屏媒体查询与支持查询共同决定布局。

特性查询不是浏览器品牌检测，也不是完整效果测试。某属性在一种布局中被识别，不意味着另一布局组合没有缺陷；例如只查询 gap 的接受情况不能独立证明旧版本的 Flexbox gap 行为。仍须检查本例实际布局。

不认识 @supports 的浏览器会忽略该块，所以关键回退必须放在块外，不应只写在 @supports not 里。

```html
<div class="cards">
  <article class="card"><h2>基础内容</h2><p>默认按文档顺序排列。</p></article>
  <article class="card"><h2>增强布局</h2><p>支持时使用两列，窄屏变成一列。</p></article>
</div>
```

```css
.cards { display: block; }
.card { border: 1px solid #778899; padding: 16px; margin-block: 12px; }
@supports (display: grid) and (gap: 12px) {
  .cards { display: grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap: 12px; }
  .card { margin: 0; }
  @media (max-width: 600px) {
    .cards { grid-template-columns: minmax(0, 1fr); }
  }
}
/* 禁用整个增强块，内容仍纵向排列；600px及以下的增强分支也只有一列。 */
```

配套文件：[enhancement.html](scripts/20-debugging-and-compatibility/enhancement.html)、[enhancement.css](scripts/20-debugging-and-compatibility/enhancement.css) · [浏览器预览](http://127.0.0.1:8101/scripts/20-debugging-and-compatibility/enhancement.html)

## 5 回退声明与支持表的读法

同一属性先写基础值、后写增强值时，后者在解析时无效通常会被忽略。这里故意用错误颜色演示恢复路径；实际工程应使用已核查的新语法，而不是保留拼写错误。

这不覆盖计算值阶段的所有失败：var() 替换后才发现值不适用时，浏览器不会重新选择前一条已被层叠淘汰的声明。需要分别检查自定义属性是否存在、值的类型是否符合使用位置。

selector(:focus-visible) 查询选择器支持，和查询属性声明是两种条件写法。基础 :focus 轮廓始终保留，增强只补充背景。

查阅 MDN 的 Browser compatibility 时，找到具体属性、值或子特性行，核对浏览器版本、移动端、部分支持、前缀和实验开关注释。不要只看页面顶部的 Baseline 标签，也不要把规范列出语法等同于所有浏览器已经实现。

确定目标版本后，在相应引擎中运行核心操作。禁用增强块能验证回退结构，却不能模拟某旧版本的所有解析和布局差异。

```html
<p class="fallback-color">无效的后写声明不会覆盖有效颜色。</p>
<a class="action-link" href="#details">跳到详细内容</a>
<section id="details"><h2>详细内容</h2><p>功能不依赖增强后的列数。</p></section>
```

```css
.fallback-color { color: navy; color: 20px; }
.action-link:focus { outline: 3px solid #a33b00; outline-offset: 3px; }
@supports selector(:focus-visible) {
  .action-link:focus-visible { background: #fff0cf; }
}
/* color:20px是刻意的解析期无效值；焦点轮廓在不支持selector()的浏览器仍存在。 */
```

配套文件：[enhancement.html](scripts/20-debugging-and-compatibility/enhancement.html)、[enhancement.css](scripts/20-debugging-and-compatibility/enhancement.css) · [浏览器预览](http://127.0.0.1:8101/scripts/20-debugging-and-compatibility/enhancement.html)

## 6 厂商前缀与废弃写法

-webkit-、-moz- 等前缀常见于历史实验或兼容实现；名字带前缀不意味着只能在该品牌浏览器生效。应依据所用功能的支持说明决定是否保留，不能批量复制所有前缀。

本例用 -webkit-appearance 与标准 appearance 示范声明顺序。appearance: none 修改控件外观，不会自动删除按钮的语义与键盘行为；失去原生视觉后，需要提供边框、文字、禁用态和焦点样式。

不要给废弃的旧布局语法简单去掉前缀，就当作迁移成功。旧版 Flexbox 写法的属性模型也可能不同，应该根据现代 display: flex 的规则重新核对布局。

工程中可由按目标版本配置的工具生成必要前缀，但产物仍需检查。本章不安装构建工具，也不把这条示范前缀当作全项目要求。

```html
<button class="styled-button" type="button">可聚焦按钮</button>
<button class="styled-button" type="button" disabled>禁用按钮</button>
<label for="email">邮箱</label><input id="email" type="email">
<p class="long-text">较长标识：hands-on-computing-css-compatibility-long-content-example</p>
```

```css
.styled-button {
  -webkit-appearance: none;
  appearance: none;
  border: 1px solid #246;
  background: #e5f0f4;
  color: #17212b;
  padding: 8px 16px;
}
.styled-button:disabled { color: #666; }
/* 前缀仅作为历史目标的示范；标准声明放后面，不代表新页面必须复制此前缀。 */
```

配套文件：[controls.html](scripts/20-debugging-and-compatibility/controls.html)、[controls.css](scripts/20-debugging-and-compatibility/controls.css) · [浏览器预览](http://127.0.0.1:8101/scripts/20-debugging-and-compatibility/controls.html)

## 7 视口、缩放与键盘检查

在设备工具栏使用响应式视口，至少检查 360px、600px、601px 和 900px；断点两侧都检查，避免只看一个典型设备预设。保留长内容，核对横向滚动、换行和控件是否被截断。

用浏览器菜单把页面缩放设为 200%，检查文字与控件。DevTools 预览画布的缩放比例不等于浏览器页面缩放；CSS transform: scale() 也不能代替该检查。有文字单独缩放能力的目标浏览器还应验证文字缩放。

从地址栏之后开始连续按 Tab、Shift+Tab，观察焦点顺序、焦点轮廓和是否被遮挡；用 Enter 或空格操作适用的原生控件。禁用按钮应跳过键盘焦点，外观灰色但未禁用的按钮则不是同一种状态。

Styles 的强制 :hover、:focus 等状态便于定位规则，但不能替代真实鼠标和键盘操作。设备模拟也不是对真实设备、输入方式和浏览器内核的完整替代。

```html
<button class="styled-button" type="button">可聚焦按钮</button>
<button class="styled-button" type="button" disabled>禁用按钮</button>
<label for="email">邮箱</label><input id="email" type="email">
<p class="long-text">较长标识：hands-on-computing-css-compatibility-long-content-example</p>
```

```css
input { max-width: 100%; box-sizing: border-box; }
.long-text { overflow-wrap: anywhere; }
@media (max-width: 600px) {
  .styled-button { display: block; margin-block: 12px; }
}
/* 在窄视口、200%页面缩放与Tab导航下检查：长文本不撑宽页面，控件焦点未被裁切。 */
```

配套文件：[controls.html](scripts/20-debugging-and-compatibility/controls.html)、[controls.css](scripts/20-debugging-and-compatibility/controls.css) · [浏览器预览](http://127.0.0.1:8101/scripts/20-debugging-and-compatibility/controls.html)

## 本章小结

- 排错按资源、匹配、语法、层叠和适用条件逐步进行。
- 计算样式、盒模型和布局覆盖层共同解释尺寸，单看声明不够。
- 基础功能放在增强之外；支持查询与支持表需要结合实际行为判断。
- 前缀按目标需求保留；窄屏、页面缩放和真实键盘操作都要检查。

## 练习

在 scripts/20-debugging-and-compatibility/ 内练习后恢复原文件。

（1）仅去掉首页的 id="notice"，预测颜色，并在 Styles 和 Computed 中定位改变原因；不要同时修改 CSS。

（2）把 .measure 改为 border-box，预测外宽与内容宽，再用盒模型图核对。

（3）禁用整个 Grid 增强块，验证两张卡片、链接和详细内容仍可用；恢复后比较 600px 与 601px。

（4）在 200% 页面缩放下只用键盘走完控件页，记录一处实际发现及其规则来源；若没有问题，如实写检查范围，不编造缺陷。

### 提示

为每个问题保存一个最小触发条件；区分开发者工具的临时变化与本地保存后的最终行为。

## 参考与引用来源

- Chrome for Developers：[CSS issues](https://developer.chrome.com/docs/devtools/css/issues) 的无效、覆盖与不活跃样式；[CSS features reference](https://developer.chrome.com/docs/devtools/css/reference) 的 Computed、盒模型和状态模拟；[Flexbox tools](https://developer.chrome.com/docs/devtools/css/flexbox)、[Grid tools](https://developer.chrome.com/docs/devtools/css/grid) 的布局覆盖层；[Network](https://developer.chrome.com/docs/devtools/network) 的请求检查；[Device mode](https://developer.chrome.com/docs/devtools/device-mode) 的响应式视口与模拟边界。
- MDN：[@supports](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/At-rules/@supports#syntax) 的声明、选择器和逻辑条件；[CSS error handling](https://developer.mozilla.org/en-US/docs/Web/CSS/Guides/Syntax/Error_handling) 的无效声明与变量替换边界；[Vendor prefix](https://developer.mozilla.org/en-US/docs/Glossary/Vendor_Prefix) 的历史用途；[appearance](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/appearance) 的原生外观、前缀与 Browser compatibility。
- Python 3.12：[http.server 命令行](https://docs.python.org/3.12/library/http.server.html#command-line-interface) 的服务参数。